# **Employee Data Profiling Overview**

This notebook performs data profiling on the Employee dataset to evaluate structure, quality, and consistency before further processing.

## Objectives

* Validate schema (columns, data types)
* Check missing values and duplicates
* Analyze data distribution (age, salary, etc.)
* Detect anomalies and inconsistencies

## Scope

* Row & column count
* Data types validation
* Null value analysis
* Distinct count (cardinality)
* Summary stats (min, max, mean, stddev)
* Basic outlier checks

## Outcome

Provides insights into data quality and readiness for cleaning, transformation, and analysis.

**Status:** Data profiling initiated...
**Author:** Ritik
**Env:** Development / Testing


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

print("Required libraries imported successfully. :) ")

Required libraries imported successfully. :) 


In [2]:
spark = (
    SparkSession.builder
    .appName("EmployeesDataProfiling")
    .getOrCreate()
)

print("SparkSession Build Successfully :) ")

SparkSession Build Successfully :) 


In [3]:
try :
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv("/content/the_data_echo/row_data/raw_employees.csv")
    )
    print(f"Data Loaded successfully row_count {df.count()}  :) ")

except Exception as e :
    print(f"Error Occurred : {e}")

Data Loaded successfully row_count 100  :) 


## **Employees First Name Data Profiling**

In [80]:
# first_name data type check
df.select("first_name").dtypes

[('first_name', 'string')]

In [99]:
# null check in first name
df.select("first_name").filter(F.col("first_name").isNull()).count()

0

In [112]:
# max length without trim
df.select(F.max(F.length(F.col("first_name"))).alias("mfn_length")).show()

+----------+
|mfn_length|
+----------+
|        12|
+----------+



In [113]:
# max length with trim
df.select(F.max(F.length(F.trim(F.col("first_name")))).alias("mfn_length")).show()

+----------+
|mfn_length|
+----------+
|         9|
+----------+



In [111]:
# min length without trim
df.select(F.min(F.length(F.col("first_name"))).alias("mfn_length")).show()

+----------+
|mfn_length|
+----------+
|         4|
+----------+



In [110]:
# min length with trim
df.select(F.min(F.length(F.trim(F.col("first_name")))).alias("mfn_length")).show()

+----------+
|mfn_length|
+----------+
|         4|
+----------+



In [96]:
df.select("first_name")\
  .withColumn("clean_first_name",  F.trim(F.initcap(F.col("first_name"))))\
    .withColumn("fn_length", F.length(F.col("first_name")))\
    .withColumn("cfn_length", F.length(F.col("clean_first_name")))\
    .filter(F.col("fn_length") != F.col("cfn_length"))\
  .show()

+------------+----------------+---------+----------+
|  first_name|clean_first_name|fn_length|cfn_length|
+------------+----------------+---------+----------+
|    Matthew |         Matthew|       10|         7|
|  Alexander |       Alexander|       12|         9|
|      Kevin |           Kevin|        8|         5|
|     Joshua |          Joshua|        9|         6|
|      Betty |           Betty|        8|         5|
+------------+----------------+---------+----------+



## **Employees Last Name Data Profiling**

In [97]:
# Last Name datatype
df.select("last_name").dtypes

[('last_name', 'string')]

In [117]:
# last name null count
df.select("last_name").filter(F.col("last_name").isNull()).count()

0

In [118]:
# last name max length
df.select(F.max(F.length(F.col("last_name"))).alias("mln_length")).show()

+----------+
|mln_length|
+----------+
|        12|
+----------+



In [119]:
# last name max length after trim
df.select(F.max(F.length(F.trim(F.col("last_name")))).alias("mln_length")).show()

+----------+
|mln_length|
+----------+
|        10|
+----------+



In [125]:
df.select("last_name")\
  .withColumn("cl_name", F.trim(F.initcap(F.col("last_name"))))\
  .withColumn("ln_length", F.length(F.col("last_name")))\
  .withColumn("cln_length", F.length(F.col("cl_name")))\
  .filter(F.col("ln_length") != F.col("cln_length"))\
  .show()

+------------+---------+---------+----------+
|   last_name|  cl_name|ln_length|cln_length|
+------------+---------+---------+----------+
|    Jimenez |  Jimenez|       10|         7|
|     Torres |   Torres|        9|         6|
|      Jones |    Jones|        8|         5|
|   Castillo | Castillo|       11|         8|
|    Sanders |  Sanders|       10|         7|
|      Gomez |    Gomez|        8|         5|
|    Mendoza |  Mendoza|       10|         7|
|      Evans |    Evans|        8|         5|
|    Stewart |  Stewart|       10|         7|
|  Hernandez |Hernandez|       12|         9|
|       Hall |     Hall|        7|         4|
|   Phillips | Phillips|       11|         8|
|       Ward |     Ward|        7|         4|
+------------+---------+---------+----------+



## **Employees Full Name Data Profiling**

In [128]:
# checking full name data type
df.select("full_name").dtypes

[('full_name', 'string')]

In [131]:
# null checking in full name
df.select("full_name").filter(F.col("full_name").isNull()).count()

0

In [ ]:
df.withColumn("max_fn_length", F.)

In [4]:
df.sample(fraction=0.1, withReplacement=False).show(truncate=False)

+-----------+----------+-----------+----------------+-------------------------------+--------------+----------------------+----------------+--------+---------------------+-------------+------------+--------------+-----------------+-------------------+----------+------------------+----------+
|employee_id|first_name|last_name  |full_name       |email                          |phone         |job_title             |department      |store_id|store_name           |store_city   |hire_date   |years_employed|annual_salary_usd|commission_rate_pct|is_active |performance_rating|manager_id|
+-----------+----------+-----------+----------------+-------------------------------+--------------+----------------------+----------------+--------+---------------------+-------------+------------+--------------+-----------------+-------------------+----------+------------------+----------+
|4019       |  Matthew |  Castillo |Matthew Castillo|matthew.castillo@hotmail.com   |589.846.2995  |Senior Sales Associat

In [ ]:
------+-------------------+---------+------------------+----------+
full_name        |email
|phone|job_title|department|store_id|store_name|store_city  |hire_date

## **Employee ID data profiling**

In [5]:
# Null check
df.select("employee_id").filter(F.col("employee_id").isNull()).count()

0

In [6]:
# Uniqueness (duplicate)
df.groupBy("employee_id").count().filter("count > 1").show()

+-----------+-----+
|employee_id|count|
+-----------+-----+
+-----------+-----+



In [7]:
# Data type validation
df.select("employee_id").dtypes

[('employee_id', 'int')]

In [8]:
# Range check
df.select("employee_id").filter(F.col("employee_id") < 1).show()

+-----------+
|employee_id|
+-----------+
+-----------+



In [9]:
df.select("employee_id").sample(fraction=0.1, withReplacement=False).show()

+-----------+
|employee_id|
+-----------+
|       4009|
|       4011|
|       4044|
|       4055|
|       4061|
|       4079|
|       4090|
|       4100|
+-----------+



## **Manager ID data Profiling**

In [10]:

df.select("first_name","manager_id").filter(F.col("manager_id").isNull()).show()

+----------+----------+
|first_name|manager_id|
+----------+----------+
|    George|      NULL|
|     Emily|      NULL|
|     Laura|      NULL|
|     James|      NULL|
|     Frank|      NULL|
|   Joshua |      NULL|
|     tyler|      NULL|
|  nicholas|      NULL|
|     Jacob|      NULL|
|   Deborah|      NULL|
|   Deborah|      NULL|
|     Joyce|      NULL|
|     Frank|      NULL|
|      Mark|      NULL|
|     Laura|      NULL|
|   CAROLYN|      NULL|
|     Diane|      NULL|
+----------+----------+



In [11]:
df.select("manager_id").filter(F.col("manager_id").isNull()).count()

17

In [12]:
df.select("manager_id").filter(F.col("manager_id").isNotNull()).count()

83

In [13]:
df.groupBy("manager_id").count().filter(F.col("manager_id").isNotNull()).filter("count > 1").show()

+----------+-----+
|manager_id|count|
+----------+-----+
|      4092|    2|
|      4061|    2|
|      4040|    2|
|      4032|    2|
|      4012|    2|
|      4037|    2|
|      4007|    2|
|      4023|    2|
|      4068|    2|
|      4020|    3|
|      4066|    2|
|      4010|    2|
|      4086|    2|
|      4055|    2|
|      4053|    2|
|      4098|    2|
|      4050|    2|
|      4003|    2|
|      4096|    2|
|      4021|    2|
+----------+-----+
only showing top 20 rows


In [14]:
df.select("manager_id").dtypes

[('manager_id', 'int')]

## **performance_rating Data Profiling**

In [15]:
df.select("performance_rating").dtypes

[('performance_rating', 'string')]

In [16]:
df.select("performance_rating").distinct().show()

+------------------+
|performance_rating|
+------------------+
|         Excellent|
|                 3|
|                 5|
|                 B|
|           Average|
|     Below Average|
|              Good|
|                 D|
|                 C|
|                 A|
|                 4|
|                 2|
|              NULL|
+------------------+



In [17]:
df.select("performance_rating").filter(F.col("performance_rating").isNotNull()).count()

94

In [18]:
df.groupBy("performance_rating").count().show()

+------------------+-----+
|performance_rating|count|
+------------------+-----+
|         Excellent|   13|
|                 3|    9|
|              NULL|    6|
|                 5|    7|
|                 B|    4|
|           Average|    4|
|     Below Average|    1|
|              Good|   10|
|                 D|   12|
|                 C|   11|
|                 A|    7|
|                 4|    4|
|                 2|   12|
+------------------+-----+



## **Employee Is active or not data profiling**

In [19]:
df.select("is_active").dtypes

[('is_active', 'string')]

In [20]:
df.select("is_active").filter(F.col("is_active").isNull()).count()

0

In [21]:
df.select("is_active").distinct().show()

+----------+
| is_active|
+----------+
|         Y|
|    Active|
|         N|
|Terminated|
|        No|
|       Yes|
|         1|
+----------+



In [22]:
df.groupBy("is_active").count().show()

+----------+-----+
| is_active|count|
+----------+-----+
|         Y|   19|
|    Active|   19|
|         N|   14|
|Terminated|   12|
|        No|   10|
|       Yes|   13|
|         1|   13|
+----------+-----+



## **Commission_rate_pct Data Profiling**

In [23]:
df.select("commission_rate_pct").dtypes

[('commission_rate_pct', 'double')]

In [24]:
df.select("commission_rate_pct").filter(F.col("commission_rate_pct").isNull()).count()

0

In [25]:
df.sample(fraction=0.1, withReplacement=False).select("commission_rate_pct").show()

+-------------------+
|commission_rate_pct|
+-------------------+
|                5.2|
|                4.5|
|                1.1|
|                4.1|
|                1.9|
|                5.4|
|                6.9|
|                4.1|
|                5.2|
|                3.1|
|                5.1|
|                5.7|
|                6.0|
|                5.3|
|                6.3|
+-------------------+



In [26]:
df.select("commission_rate_pct").filter(F.col("commission_rate_pct") < 0).show()

+-------------------+
|commission_rate_pct|
+-------------------+
+-------------------+



In [27]:
df.select("commission_rate_pct").filter(F.col("commission_rate_pct") > 100).show()

+-------------------+
|commission_rate_pct|
+-------------------+
+-------------------+



In [28]:
df.select(F.max("commission_rate_pct").alias("max_cmmission_rate")).show()

+------------------+
|max_cmmission_rate|
+------------------+
|               7.9|
+------------------+



In [29]:
df.select(F.min("commission_rate_pct").alias("min_cmmission_rate")).show()

+------------------+
|min_cmmission_rate|
+------------------+
|               1.1|
+------------------+



In [30]:
df.select(F.mode("commission_rate_pct").alias("mode_commission_rate")).show()

+--------------------+
|mode_commission_rate|
+--------------------+
|                 2.6|
+--------------------+



In [31]:
df.select("commission_rate_pct").summary().show()

+-------+-------------------+
|summary|commission_rate_pct|
+-------+-------------------+
|  count|                100|
|   mean|              4.326|
| stddev| 1.9599845392092743|
|    min|                1.1|
|    25%|                2.6|
|    50%|                4.2|
|    75%|                5.9|
|    max|                7.9|
+-------+-------------------+



## **Employee annual_salary_usd Data Profiling**

In [32]:
df.select("annual_salary_usd").show(10)

+-----------------+
|annual_salary_usd|
+-----------------+
|            75248|
|            84780|
|           116056|
|            93447|
|            92114|
|            76680|
|            40962|
|            63122|
|            36973|
|           102209|
+-----------------+
only showing top 10 rows


In [33]:
df.select("annual_salary_usd").dtypes

[('annual_salary_usd', 'int')]

In [34]:
df.select("annual_salary_usd").filter(F.col("annual_salary_usd").isNull()).count()

0

In [35]:
df.select("annual_salary_usd").filter(F.col("annual_salary_usd") < 0).count()

0

In [40]:
df.select("annual_salary_usd")\
.agg(F.mean(F.col("annual_salary_usd")).alias("min_annual_salary_usd")).show(truncate=False)

+---------------------+
|min_annual_salary_usd|
+---------------------+
|75849.64             |
+---------------------+



In [37]:
df.groupBy("department")\
  .agg(F.round(F.mean(F.col("annual_salary_usd"))).alias("mean_annual_salary_usd"))\
  .orderBy(F.col("mean_annual_salary_usd").desc()).show()

+----------------+----------------------+
|      department|mean_annual_salary_usd|
+----------------+----------------------+
|      Operations|               82373.0|
|Customer Service|               74699.0|
|           Sales|               74171.0|
|      Management|               72644.0|
+----------------+----------------------+



In [90]:
df.groupBy("department")\
  .agg(F.max(F.col("annual_salary_usd")).alias("max_department_salary"),
       F.round(F.mean(F.col("annual_salary_usd")),2).alias("mean_department_salary"),
       F.min(F.col("annual_salary_usd")).alias("min_salary"),
       F.mode(F.col("annual_salary_usd")).alias("mode_salary"),
       F.var_samp("annual_salary_usd").alias("variance_salary"),
       F.stddev("annual_salary_usd").alias("stdv_salary")
       )\
  .orderBy(F.col("max_department_salary").desc())\
  .show()

+----------------+---------------------+----------------------+----------+-----------+-------------------+------------------+
|      department|max_department_salary|mean_department_salary|min_salary|mode_salary|    variance_salary|       stdv_salary|
+----------------+---------------------+----------------------+----------+-----------+-------------------+------------------+
|Customer Service|               118605|              74699.12|     36973|      55019|6.402483187130126E8|25303.128634874633|
|           Sales|               118067|              74171.17|     32165|     102022|7.765478914492755E8|27866.608897554714|
|      Operations|               116056|              82373.23|     38948|      76519|6.079098750411255E8|24655.828419283047|
|      Management|               113221|              72643.75|     33272|      74929|6.655939216710526E8|25799.106993674268|
+----------------+---------------------+----------------------+----------+-----------+-------------------+------------

## **years_employed Data customer Profiling**

In [58]:
df.select("years_employed").dtypes

[('years_employed', 'int')]

In [59]:
df.select("years_employed").filter(F.col("years_employed").isNull()).count()

0

In [61]:
df.select("years_employed")\
  .agg(
      F.max(F.col("years_employed")).alias("max_years_employed"),
      F.round(F.mean(F.col("years_employed")),2).alias("mean_years_employed")
  ).show()

+------------------+-------------------+
|max_years_employed|mean_years_employed|
+------------------+-------------------+
|                14|               7.03|
+------------------+-------------------+



In [64]:
df.createOrReplaceTempView("customers")

In [74]:
spark.sql("""
      SELECT
        MAX(years_employed) AS max_years_employed,
        MEDIAN(years_employed) AS median_years_employed,
        MODE(years_employed) AS mode_years_employed,
        MEAN(years_employed) AS avg_years_employed,
        MIN(years_employed) AS min_years_employed
      FROM customers
""").show()

+------------------+---------------------+-------------------+------------------+------------------+
|max_years_employed|median_years_employed|mode_years_employed|avg_years_employed|min_years_employed|
+------------------+---------------------+-------------------+------------------+------------------+
|                14|                  7.0|                 13|              7.03|                 0|
+------------------+---------------------+-------------------+------------------+------------------+



In [79]:
df.select("years_employed").filter(F.col("years_employed") == 13).count()

11